# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4.1`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

In [5]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os

from agent_framework import tool
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import DefaultAzureCredential

client = FoundryChatClient(
    credential=DefaultAzureCredential(),
    project_endpoint=os.getenv("AZURE_AI_PROJECT_ENDPOINT"),
    model=os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
)

## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [6]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [7]:
agent = Agent(
    client=client,
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

Based on popular vacation destinations, here are some warm beach locations I recommend:

1. Bali – Known for its beautiful beaches, vibrant culture, and warm weather year-round.
2. Sydney – Offers stunning coastal beaches like Bondi and Manly, plus a lively city atmosphere.
3. Cape Town – Famous for its scenic beaches set against dramatic landscapes.
4. Rio de Janeiro – Home to iconic beaches like Copacabana and Ipanema, with a tropical climate.

Would you like more information on any of these destinations or help narrowing it down based on activities, budget, or travel dates?


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [8]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

Tokyo is one of is one of the world’s most exciting and dynamic cities, known for its stunning blend of traditional culture and cutting-edge modernity. Here’s what you can expect from Tokyo as a travel destination:

**Highlights:**
- **Culture & History:** Tokyo boasts spectacular historic sites such as the Senso-ji Temple in Asakusa, the Meiji Shrine in Shibuya, and tranquil Japanese gardens.
- **Food:** Tokyo is a food lover’s paradise, with everything from Michelin-starred sushi and ramen to quirky themed cafés and bustling night markets.
- **Shopping & Entertainment:** Explore famous districts like Shibuya and Shinjuku for shopping, vibrant nightlife, and entertainment; Tokyo Disneyland and DisneySea are also popular attractions.
- **Technology & Innovation:** See the latest in fashion, robotics, and tech in districts like Akihabara, the hub of electronics and anime culture.
- **Seasonal Beauty:** Cherry blossom season in the spring transforms the city’s parks, while autumn brings 

## Summary

In this lesson you learned how to:

- **Create an agent** using the `FoundryChatClient`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.